# 04 · Approach 2: Bidirectional LSTM/GRU with attention
Owner: **Member 2** · Runtime: Colab **GPU** (T4)

**Why this model.** A recurrent network reads the article word by word and carries a memory of
what it has read. Each word is interpreted in the context of everything before it (forward
direction) and after it (backward direction). Gates (LSTM: Hochreiter & Schmidhuber, 1997; GRU:
Cho et al., 2014) let that memory survive across hundreds of words, which matters here because the
median article is ~280 words long. **Attention pooling** (Bahdanau et al., 2015; Yang et al., 2016)
learns which words decide the category instead of relying only on the last hidden state, and its
weights can be inspected.
**Limitations:** computation is sequential and slow on long inputs, and memory still fades over very
long distances.

## 0 · Setup
Run this cell first. On **Google Colab** it clones the repo, installs requirements and copies the
data from your Google Drive. Locally it does nothing special.

**Before the first run:** put `Train.csv`, `Test.csv` and `SampleSubmission.csv` (from Zindi) in a
Google Drive folder called `swahili_news`, and set `REPO_URL` below.

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_URL = "https://github.com/YOUR-GROUP/swahili-news-sequential.git"   # <-- edit once
    REPO_DIR = Path("/content/swahili-news-sequential")
    DRIVE_DATA = Path("/content/drive/MyDrive/swahili_news")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    for name in ("Train.csv", "Test.csv", "SampleSubmission.csv"):
        shutil.copy(DRIVE_DATA / name, REPO_DIR / "data" / name)
    os.environ["CACHE_DIR"] = str(DRIVE_DATA / "cache")             # e.g. downloaded word vectors persist
    os.environ["OUTPUT_BACKUP_DIR"] = str(DRIVE_DATA / "outputs")   # results backed up to Drive
else:
    if Path.cwd().name == "notebooks":
        os.chdir(Path.cwd().parent)
sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import HTML, display

from src import config as C
from src.data import get_splits, split_frames, encode_labels, load_unlabelled
from src.datasets import TokenDataset, pad_collate
from src.models import BiRNNAttention
from src.pipeline import run_experiments, finalize
from src.text import tokenize, head_tail, Vocab, encode_corpus, load_fasttext_vectors
from src.utils import set_seed, get_device

MEMBER, MODEL = "Member 2", "BiRNN attention"
set_seed(C.SEED)
print("Device:", get_device())
df = get_splits()
tr, va, te = split_frames(df)
ytr, yva, yte = (encode_labels(d.label) for d in (tr, va, te))
zindi = load_unlabelled()
K = len(C.CLASSES)
vocab = Vocab(tr.content.map(tokenize))

_cache = {}


def seqs(max_len, head_frac=1.0):
    key = (max_len, head_frac)
    if key not in _cache:
        _cache[key] = {n: encode_corpus(d.content, vocab, max_len, head_frac) for n, d in
                       {"tr": tr, "va": va, "te": te, "z": zindi}.items()}
    return _cache[key]


# pretrained Swahili fastText vectors, if notebook 03 (or you) already downloaded them to the cache
vec_path = Path(C.CACHE_DIR) / "cc.sw.300.vec.gz"
EMB = load_fasttext_vectors(vocab, vec_path, 300)[0] if vec_path.exists() else None
print("Pretrained vectors available:", EMB is not None)


def make_datasets(p):
    s = seqs(p.get("max_len", C.MAX_WORDS), p.get("head_frac", 1.0))
    return TokenDataset(s["tr"], ytr, p.get("word_dropout", 0.0)), TokenDataset(s["va"], yva)


def build_model(p):
    pre = EMB if p.get("pretrained") else None
    return BiRNNAttention(len(vocab), K, dim=300 if pre is not None else 128, hidden=p.get("hidden", 128),
                          layers=p.get("layers", 1), rnn=p.get("rnn", "lstm"), pooling=p.get("pooling", "attention"),
                          bidirectional=p.get("bidirectional", True), dropout=p.get("dropout", 0.4), pretrained=pre)

## Experiments

In [ ]:
DEFAULT_TRAIN = dict(epochs=20, lr=2e-3, batch_size=32, weight_decay=1e-4, patience=4, num_workers=0)
EXPERIMENTS = [
    dict(id="RNN-01", description="BiLSTM (128 per direction), masked mean pooling, 512 words",
         hypothesis="Reading the sequence with memory beats order-free baselines.",
         motivated_by="Starting point (compare with BL-02 and FT-02)", params={"pooling": "mean"}),
    dict(id="RNN-02", description="BiLSTM + additive attention pooling",
         hypothesis="Attention picks out the few decisive words in long articles; mean pooling dilutes them.",
         motivated_by="EDA 4: median ~280 words; RNN-01 errors on long articles", params={"pooling": "attention"}),
    dict(id="RNN-03", description="BiGRU + attention",
         hypothesis="GRU's simpler gating trains as well with fewer parameters on a small dataset.",
         motivated_by="Gating comparison (Chung et al., 2014 found no universal winner)", params={"rnn": "gru"}),
    dict(id="RNN-04", description="Unidirectional LSTM + attention",
         hypothesis="Backward context matters: later sentences clarify earlier ones.",
         motivated_by="Ablation of bidirectionality from RNN-02", params={"bidirectional": False}),
    dict(id="RNN-05", description="BiLSTM + attention, head+tail truncation (first 384 + last 128 words)",
         hypothesis="Keeping the end of long articles recovers signal lost by head truncation.",
         motivated_by="EDA 8: keywords are only mildly front-loaded; BL-05 tail-only result",
         params={"head_frac": 0.75}),
    dict(id="RNN-06", description="BiLSTM + attention + class weights + word dropout",
         hypothesis="Re-weighting and regularisation improve minority-class recall without hurting log loss.",
         motivated_by="RNN-02 per-class F1 on kimataifa/burudani + overfitting in learning curves",
         params={"word_dropout": 0.1}, train={"class_weights": True}),
]
if EMB is not None:
    EXPERIMENTS.append(dict(id="RNN-07", description="BiLSTM + attention with pretrained fastText vectors",
                            hypothesis="Pretrained vectors help the RNN more than the CNN, since it has more parameters to fit.",
                            motivated_by="CNN-06 result + EDA 5 (OOV/hapax)", params={"pretrained": True}))
best, summary = run_experiments(EXPERIMENTS, build_model, make_datasets, member=MEMBER, model_name=MODEL,
                                n_classes=K, y_train=ytr, default_train=DEFAULT_TRAIN, collate_fn=pad_collate)
summary.round(4)

> Fill the `observation` column in `experiments/log_birnn_attention.csv`: attention vs mean pooling,
> LSTM vs GRU, uni- vs bidirectional, head vs head+tail. Add follow-ups if something surprises you.

## Final evaluation on the test split (once)

In [ ]:
p = best["exp"]["params"]
s = seqs(p.get("max_len", C.MAX_WORDS), p.get("head_frac", 1.0))
model, proba_test, metrics = finalize(best, build_model, TokenDataset(s["te"], yte), yte, te.id.values, C.CLASSES,
                                      key="birnn_attention", display_name="BiLSTM/GRU + attention",
                                      collate_fn=pad_collate, zindi_ds=TokenDataset(s["z"]), zindi_df=zindi)

## What does the model attend to?
Words highlighted by attention weight for a few correct and a few wrong test articles. Use these in
the error analysis: does attention land on topic words, on names, or on noise?

In [ ]:
rows = []
if p.get("pooling", "attention") == "attention":
    dev = get_device()
    model.eval().to(dev)
    pred = proba_test.argmax(1)
    pick = list(np.where(pred == yte)[0][:2]) + list(np.where(pred != yte)[0][:4])
    for i in pick:
        toks = head_tail(tokenize(te.content[i]), p.get("max_len", C.MAX_WORDS), p.get("head_frac", 1.0))
        ids = torch.tensor([vocab.encode(toks)]).to(dev)
        with torch.no_grad():
            model(input_ids=ids, lengths=torch.tensor([len(toks)]))
        a = model.last_attention[0].cpu().numpy()[:len(toks)]
        top = [toks[j] for j in np.argsort(-a)[:10]]
        rows.append({"id": te.id[i], "true": C.CLASSES[yte[i]], "pred": C.CLASSES[pred[i]], "top_attended_words": ", ".join(top)})
        html = " ".join(f'<span style="background: rgba(255,120,0,{min(1, w / a.max()):.2f})">{t}</span>'
                        for t, w in zip(toks[:150], a[:150]))
        display(HTML(f"<b>true={C.ENGLISH[C.CLASSES[yte[i]]]} | pred={C.ENGLISH[C.CLASSES[pred[i]]]}</b><br>{html} ...<hr>"))
    pd.DataFrame(rows).to_csv(C.RESULTS_DIR / "birnn_attention" / "attention_examples.csv", index=False)
pd.DataFrame(rows)

In [ ]:
from src.utils import persist_outputs
persist_outputs()